In [0]:
%sql
-- Create reference table for stadium timezone and country mapping
-- This is a static lookup table for the 16 host stadiums in FIFA World Cup 2026
-- VERIFIED: Mexico does NOT observe DST (uses standard time year-round)
-- MOVED TO SILVER LAYER: One-time static reference table (not managed by dbt)

CREATE OR REPLACE TABLE singhal.fifa_worldcup_silver.ref_stadium_enriched (
  stadium_id STRING,
  city STRING,
  actual_country STRING,
  timezone_abbr STRING,
  utc_offset_hours INT
)
COMMENT 'Reference table: Stadium timezone and country mapping for UTC conversion';

-- Insert all 16 stadiums with VERIFIED correct timezones
INSERT INTO singhal.fifa_worldcup_silver.ref_stadium_enriched VALUES
  -- Mexico stadiums (NO DST - Standard Time year-round)
  ('1', 'Mexico City', 'Mexico', 'CST', 6),      -- Central Standard Time UTC-6
  ('2', 'Guadalajara', 'Mexico', 'MST', 7),      -- Mountain Standard Time UTC-7
  ('3', 'Monterrey', 'Mexico', 'CST', 6),        -- Central Standard Time UTC-6
  
  -- USA - Central Time (Observes DST in summer)
  ('4', 'Dallas', 'USA', 'CDT', 5),
  ('5', 'Houston', 'USA', 'CDT', 5),
  ('6', 'Kansas City', 'USA', 'CDT', 5),
  
  -- USA - Eastern Time (Observes DST in summer)
  ('7', 'Atlanta', 'USA', 'EDT', 4),
  ('8', 'Miami', 'USA', 'EDT', 4),
  ('9', 'Boston', 'USA', 'EDT', 4),
  ('10', 'Philadelphia', 'USA', 'EDT', 4),
  ('11', 'New York', 'USA', 'EDT', 4),
  
  -- Canada - Eastern Time (Observes DST in summer)
  ('12', 'Toronto', 'Canada', 'EDT', 4),
  
  -- Canada - Pacific Time (Observes DST in summer)
  ('13', 'Vancouver', 'Canada', 'PDT', 7),
  
  -- USA - Pacific Time (Observes DST in summer)
  ('14', 'Seattle', 'USA', 'PDT', 7),
  ('15', 'San Francisco', 'USA', 'PDT', 7),
  ('16', 'Los Angeles', 'USA', 'PDT', 7);

## 🔄 Dynamic UTC Conversion

**How it works:**

1. **Reference Table (Static)**: `ref_stadium_enriched` stores timezone offsets for each stadium
   - These are **static** - stadium timezones don't change
   - Mexico: CST/MST (no DST)
   - USA/Canada: EDT/CDT/PDT (with DST)

2. **Match Times (Dynamic)**: `silver_matches.match_date_local` contains local match times
   - If source data updates (reschedule, time change), it's reflected here

3. **UTC Conversion (Computed)**: Gold tables will compute UTC on-the-fly:
   ```sql
   match_datetime_local_parsed + MAKE_DT_INTERVAL(0, utc_offset_hours, 0, 0) AS match_datetime_utc
   ```
   - ✅ **Automatically updates** when `silver_matches` changes
   - ✅ No manual refresh needed
   - ✅ Always accurate

**Result**: If a match time changes in the silver layer, the gold layer UTC time will automatically reflect the change on next query/refresh.